In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

In [2]:
MLFLOW_TRACKING_URI = "http://mlflow:5000"
EXPERIMENT_NAME = "fraud_detection"
MODEL_NAME = "fraud_detector"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1780951881429, experiment_id='1', last_update_time=1780951881429, lifecycle_stage='active', name='fraud_detection', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [3]:
DATA_PATH = "/workspace/data/creditcard.csv"

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Fraud rate train:", y_train.mean())

Train: (227845, 30)
Test: (56962, 30)
Fraud rate train: 0.001729245759178389


In [4]:
rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

param_dist = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 5, 8, 12, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", "log2", None],
}

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=4,
    scoring="average_precision",
    cv=2,
    random_state=42,
    n_jobs=1,
    verbose=3,
)

In [5]:
with mlflow.start_run(run_name="random_forest_hyperparameter_search") as run:
    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]

    metrics = {
        "auc_pr": average_precision_score(y_test, y_prob),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "precision_fraud": precision_score(y_test, y_pred, zero_division=0),
        "recall_fraud": recall_score(y_test, y_pred),
        "f1_fraud": f1_score(y_test, y_pred),
        "best_cv_auc_pr": search.best_score_,
    }

    mlflow.log_params(search.best_params_)
    mlflow.log_param("model_type", "random_forest")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("dataset_path", DATA_PATH)

    # artifact: curva PR
    import matplotlib.pyplot as plt
    from sklearn.metrics import PrecisionRecallDisplay

    fig, ax = plt.subplots(figsize=(7, 5))
    PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=ax)
    ax.set_title("Precision-Recall Curve")

    pr_curve_path = "/workspace/notebooks/pr_curve.png"
    fig.savefig(pr_curve_path, bbox_inches="tight")
    plt.close(fig)

    mlflow.log_artifact(pr_curve_path, artifact_path="plots")
    
    mlflow.log_metrics(metrics)

    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
    )

    print("Run ID:", run.info.run_id)
    print("Best params:", search.best_params_)
    print("Metrics:", metrics)

2026/06/08 21:00:14 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Fitting 2 folds for each of 4 candidates, totalling 8 fits
[CV 1/2] END max_depth=3, max_features=None, min_samples_leaf=5, min_samples_split=5, n_estimators=200;, score=0.591 total time=  19.3s
[CV 2/2] END max_depth=3, max_features=None, min_samples_leaf=5, min_samples_split=5, n_estimators=200;, score=0.656 total time=  18.3s
[CV 1/2] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=300;, score=0.839 total time=  12.8s
[CV 2/2] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=300;, score=0.829 total time=  13.1s
[CV 1/2] END max_depth=8, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=200;, score=0.816 total time=   6.6s
[CV 2/2] END max_depth=8, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=200;, score=0.805 total time=   6.6s
[CV 1/2] END max_depth=3, max_features=None, min_samples_leaf=5, min_samples_split=10, n_estimators=200;, score=0.591 t

2026/06/08 21:02:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 21:02:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'fraud_detector' already exists. Creating a new version of this model...
2026/06/08 21:02:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud_detector, version 2


Run ID: 28fdad4e9e74417d8d4a6b9a4bff3293
Best params: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
Metrics: {'auc_pr': 0.8697794409504176, 'roc_auc': 0.9656361436668085, 'precision_fraud': 0.927710843373494, 'recall_fraud': 0.7857142857142857, 'f1_fraud': 0.850828729281768, 'best_cv_auc_pr': np.float64(0.8342955491482438)}
🏃 View run random_forest_hyperparameter_search at: http://mlflow:5000/#/experiments/1/runs/28fdad4e9e74417d8d4a6b9a4bff3293
🧪 View experiment at: http://mlflow:5000/#/experiments/1


Created version '2' of model 'fraud_detector'.


# Ver modelos

In [6]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.auc_pr DESC"],
)

runs[[
    "run_id",
    "metrics.auc_pr",
    "metrics.roc_auc",
    "metrics.recall_fraud",
    "metrics.precision_fraud",
    "metrics.f1_fraud",
    "params.n_estimators",
    "params.max_depth",
    "params.min_samples_leaf",
]]

,run_id,metrics.auc_pr,metrics.roc_auc,metrics.recall_fraud,metrics.precision_fraud,metrics.f1_fraud,params.n_estimators,params.max_depth,params.min_samples_leaf
0,28fdad4e9e74417d8d4a6b9a4bff3293,0.869779,0.965636,0.785714,0.927711,0.850829,300,None,1
1,dbce5b9b9e6745efb68cbe48fee8ff5e,0.869779,0.965636,0.785714,0.927711,0.850829,300,None,1
2,a978524700774160bf693925cedcf470,0.845956,0.999558,0.764706,0.866667,0.812500,100,6,None
3,e2ee73df2281493ba763da37b3ca0f57,NaN,NaN,NaN,NaN,NaN,None,None,None
4,42d4294e6f14459e85c0927cfefffb9b,NaN,NaN,NaN,NaN,NaN,None,None,None
5,5ad95f544c044f318fa1cfd734eabaa2,NaN,NaN,NaN,NaN,NaN,None,None,None
6,07d89de7398648ff9a029cd7935f340b,NaN,NaN,NaN,NaN,NaN,None,None,None


In [7]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

versions = client.search_model_versions(f"name='{MODEL_NAME}'")

for v in versions:
    print({
        "version": v.version,
        "run_id": v.run_id,
        "status": v.status,
    })

{'version': '2', 'run_id': '28fdad4e9e74417d8d4a6b9a4bff3293', 'status': 'READY'}
{'version': '1', 'run_id': 'a978524700774160bf693925cedcf470', 'status': 'READY'}


In [8]:
model_uri = f"models:/{MODEL_NAME}@champion"

loaded_model = mlflow.sklearn.load_model(model_uri)

loaded_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](30,)","['Time','V1','V2',...,'V27','V28','Amount']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
